In [3]:
import joblib
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def fundir_tres_scalers(caminho1, caminho2, caminho3, caminho_saida="minmax_scaler.pkl"):
    print("[*] A carregar os três scalers individuais...")
    # 1. Carregar os scalers originais das três bases de dados
    scaler1 = joblib.load(caminho1)
    scaler2 = joblib.load(caminho2)
    scaler3 = joblib.load(caminho3)
    
    # 2. Criar uma nova instância vazia que será o nosso Scaler Global
    scaler_global = MinMaxScaler()
    
    # 3. Encontrar matematicamente o menor mínimo e o maior máximo para cada característica
    scaler_global.data_min_ = np.minimum(np.minimum(scaler1.data_min_, scaler2.data_min_), scaler3.data_min_)
    scaler_global.data_max_ = np.maximum(np.maximum(scaler1.data_max_, scaler2.data_max_), scaler3.data_max_)
    
    # 4. Recalcular o intervalo de dados global (data_range_)
    scaler_global.data_range_ = scaler_global.data_max_ - scaler_global.data_min_
    
    # 5. Definir o intervalo padrão do Scikit-Learn (0, 1)
    feature_range = (0, 1)
    range_diff = feature_range[1] - feature_range[0]
    
    # 6. Recalcular as variáveis matemáticas internas que o transform() usa (scale_ e min_)
    # O np.where previne erros de divisão por zero se o mínimo e o máximo de uma coluna forem iguais
    scaler_global.scale_ = np.where(scaler_global.data_range_ == 0, 1.0, range_diff / scaler_global.data_range_)
    scaler_global.min_ = feature_range[0] - scaler_global.data_min_ * scaler_global.scale_
    
    # Somar o total de amostras processadas (opcional, apenas para manter metadados corretos)
    scaler_global.n_samples_seen_ = scaler1.n_samples_seen_ + scaler2.n_samples_seen_ + scaler3.n_samples_seen_
    
    # 7. Guardar o novo scaler combinado pronto para produção
    joblib.dump(scaler_global, caminho_saida)
    print(f"[+] Sucesso! Scaler global unificado guardado em: '{caminho_saida}'")

# Exemplo de uso (substitua pelos nomes reais dos seus ficheiros):
fundir_tres_scalers(
    caminho1="scaler_UNSW.pkl", 
    caminho2="scaler_BOT.pkl", 
    caminho3="scaler_CIC.pkl"
)

[*] A carregar os três scalers individuais...
[+] Sucesso! Scaler global unificado guardado em: 'minmax_scaler.pkl'
